In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
import os
import sys
file_path = os.getcwd()
project_root = os.path.dirname(file_path)
print(project_root)
# Add the project root to the sys.path
if project_root not in sys.path:
    sys.path.append(project_root)
from dotenv import load_dotenv
# from langchain.agents import AgentExecutor  # legacy (langchain 0.x; not used — agents come from crunch_llm.code.agents)
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
# from langchain.agents import create_tool_calling_agent  # legacy (langchain 0.x; not used — agents come from crunch_llm.code.agents)
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.prompts import PromptTemplate
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_core.tools import tool
from typing import Dict, List, Union, Any 
from pathlib import Path
from IPython.display import display, Image
from typing import Optional,Dict, List, Tuple  
# Encoding
import base64
import mimetypes
import io  
import PIL.Image
import json 
import subprocess
import shutil # 
import re
from datetime import datetime 
import os
from pathlib import Path
# Our Libraries
from crunch_llm.code.agents import *
from crunch_llm.code.teams_SciML_Operators import *
from crunch_llm.code.utils import *

/users/jdtoscan/.conda/envs/Agent_Hub/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/oscar/data/gk/jdtoscan/LLM/Crunch_LLM


# Initialization

In [2]:
my_database_path = "/users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Documents/database_Operators/"
my_database_path_functions="/users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Documents/database_Modules_google"
my_database_path_functions_ollama="/users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Documents/database_Modules_ollama"
database_name = 'Layers'
codes_database = 'Operators'

In [3]:
naming_agent = StructuredAgent(model='gpt-oss:120b', 
                     verbose=False,
                     system_prompt=naming_agent_system_prompt_PINN,
                    )
retriever_agent = Retriever_Agent(model='gpt-oss:120b', 
                     verbose=False,
                     system_prompt=Retriever_system_prompt_ON,
                     db_path=my_database_path,
                     collection_name=codes_database,
                     k=3)
collector_agent = Retriever_Agent(model='gpt-oss:120b', 
                     verbose=False,
                     system_prompt=collector_system_prompt_ON,
                     db_path=my_database_path_functions_ollama,
                     collection_name=database_name,
                     k=1)


librarian_agent = memoryAgent(model='gemini-2.5-pro', 
                     verbose=False,
                     system_prompt=librarian_system_prompt_ON,
                     model_provider='google',     
                    )

planner_agent = memoryAgent(model='gemini-2.5-flash', 
                     verbose=False,
                     system_prompt=planner_agent_system_prompt_ON,
                     model_provider='google',     
                    )
                    
patcher_agent = StructuredAgent(
                    model='gpt-oss:120b', 
                    system_prompt=patcher_agent_system_prompt_PINN,
                    verbose=False
                )

debugger_agent = StructuredAgent(
                    model='gemini-2.5-flash', 
                     model_provider='google',     
                    system_prompt=debugger_system_prompt_PINN,
                    verbose=False
                )
advisor_agent = aRAGAgent_multimodal(
                    model='gemini-2.5-pro', 
                    verbose=True,
                    system_prompt=advisor_system_prompt_ON,
                    model_provider='google',     
                    embeddings_provider='google', 
                    embeddings_model='models/text-embedding-004',
                    db_path=my_database_path_functions,
                    collection_name=database_name,
                    k=5
                )
advisor_parser = StructuredAgent(
                    model='gpt-oss:120b', 
                     verbose=False,
                     system_prompt=advisor_parser_system_prompt_ON,
                    )

strategist_agent = aRAGAgent_single_chunk(model='gemini-2.5-flash', 
                     verbose=False,
                     system_prompt=strategist_agent_system_prompt_ON,
                     model_provider='google',     
                    embeddings_provider='google', 
                    embeddings_model='models/text-embedding-004',
                     db_path=my_database_path_functions,
                     collection_name=database_name,
                     k=2)
register_agent_parser = StructuredAgent(
                    model='gpt-oss:120b', 
                     verbose=False,
                     system_prompt=register_agent_system_prompt_ON,
                    )

## User Query

In [4]:
user_question = """
Lets solve the viscous burgers equation the domain in x \in[0,1] and t\in [0,1] with periodic boundary conditions. Here $nu=1/(1000)$.

# Problem
This will be an operator network that maps the initial condition to the full solution up to t=t_end (extracted from the data)

### Reference Data: 
You can find the data here:
/users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Data/Operator_Network_Examples/Burger_1000_5000.mat
"""

In [5]:
checkpoint_project_dir=''
strategist_report, error_report, history_log_string, history, file_content, filename= load_checkpoint_data(checkpoint_project_dir)


--- Starting a new run (no resume directory specified). ---


In [6]:
# %%
max_iterations_all=100
test_image_summary=False
refinement_iterations=0
is_approved = False
base_results_dir='/users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Results/DeepONet_codes'
specific_requests=''#'\nSPECIFIC REQUESTS: QR-DeeNet with KKAN and RBF trained for 200000 iterations\n'# Pleaceholder for extra instructions
new_code_details='Test'
clean_functions=''
my_functions=''
requires_name=True
debug_approval=True
# --- Initialize history and log string OUTSIDE the loop ---
project_dir = "" # Initialize project_dir
while (is_approved == False and refinement_iterations < max_iterations_all):
        print(f"\n--- Code Creation Iteration: {refinement_iterations} ---")
        
        # --- Print the history from the LAST completed run ---
        print("\n--- CURRENT TRIAL HISTORY ---")
        print(history_log_string)
        print("---------------------------\n")

        filename,file_content,strategist_report,clean_functions,my_functions= code_creation_team_ON(
                                                                        user_question=user_question+specific_requests,
                                                                        history_log_string=history_log_string,
                                                                        initial_code=file_content,
                                                                        filename=filename,
                                                                        error_report=error_report,
                                                                        strategist_report=strategist_report,
                                                                        strategist_agent=strategist_agent,
                                                                        collector_agent=collector_agent,
                                                                        librarian_agent=librarian_agent,
                                                                        retriever_agent=retriever_agent,
                                                                        planner_agent=planner_agent,
                                                                        patcher_agent=patcher_agent,
                                                                        debug_approval=debug_approval,
                                                                        clean_functions=clean_functions,
                                                                        my_functions=my_functions,
                                                                        )  
        
        project_dir,filename,final_filepath,new_code_details= filing_team_PINN(user_question,
                                                                                filename,
                                                                                file_content,
                                                                                naming_agent,
                                                                                requires_name,
                                                                                new_code_details,
                                                                                base_results_dir=base_results_dir)
        requires_name=False 
        success,log_file_name=run_code(project_dir,filename,final_filepath)      
        
        status, debugger_report,debug_approval,full_log_for_agent,log_content=debugging_team_PINN(user_question,
                                                                                        final_filepath,
                                                                                        log_file_name,
                                                                                        debugger_agent)         
        
        if debug_approval:
            specific_requests='' 
            
            error_report, is_approved, run_summary = advisor_team_ON(
                                                        user_question=user_question,
                                                        history_log_string=history_log_string, 
                                                        project_dir=project_dir,
                                                        log_content=log_content,
                                                        strategist_report=strategist_report,
                                                        advisor_agent=advisor_agent,
                                                        advisor_parser=advisor_parser,
                                                        register_agent_parser=register_agent_parser,
                                                        max_width=500,
                                                        )
            history.append(run_summary) 
            history_log_string = update_and_save_report(
                history=history,
                user_question=user_question,
                strategist_report=strategist_report,
                error_report=error_report, # This is the advisor's report
                project_dir=project_dir
            )
            print("--- Clearing agent short-term memories ---")
            strategist_agent.clear_history()
            retriever_agent.clear_history()
            collector_agent.clear_history()
            librarian_agent.clear_history()
            advisor_agent.clear_history()
 
            if run_summary.get('change_architecture') == True:
                print("--- Gate Check: change_architecture is TRUE. Re-initializing file_content to force new template retrieval. ---")
                file_content='' 
        else:
            error_report=f"Debugger Analysis:\n{debugger_report}\n\n"+f"Log File:\n{full_log_for_agent}\n"        
        
        refinement_iterations=refinement_iterations+1


--- Code Creation Iteration: 0 ---

--- CURRENT TRIAL HISTORY ---
No trials have been run yet.
---------------------------

--- 1. Calling Strategist Agent ---
--- Strategy Report Received ---
## Scientific Strategy Report: Viscous Burgers Equation

### 0. Triage - Strategy Determination

This is the initial run for the Viscous Burgers Equation problem. There are no specific requests in the `[USER_QUESTION]`, and no `[ADVISOR_REPORT]` or `[TRIAL_HISTORY_LOG]` from previous runs. Therefore, we will proceed with **Strategy 1: Baseline DeepONet (MLP)**.

### 1. Data Source

The data for this problem is located at:
`/users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Data/Operator_Network_Examples/Burger_1000_5000.mat`

The Planner should load this `.mat` file and inspect its contents to identify the initial conditions (`u0`), spatial coordinates (`x`), temporal coordinates (`t`), and the full solution (`u(t,x)`).

### 2. Problem Statement

This report addresses the solution of the **viscous Bur